### function to write extra rates from Broekgaarden (2022)

the function below collects all the rates for all 560 models from Broekgaarden et al. (2022) and writes them to one big file containing just the required data (CE and wihtout CE fractions and total rate) that can then be added to the main rate comparison and formation channel comparison plots 

the code below generates a seperate file for BHBH, BHNS, and NSNS 

# main function 

In [13]:
import os
import glob
import pandas as pd
import numpy as np

# --------------------------------------------------
# Paths
# --------------------------------------------------
input_dir = "/Users/floorbroekgaarden/Projects/GitHub/DCO_FormationChannels/dataFiles/data_Fig_1"
output_dir = "/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/plottingCode/Fig_comparison/Data_formation_channels_intrinsic/extra_fc_rates"

# --------------------------------------------------
# Column names in the input files
# --------------------------------------------------
col_model = "formation channel"
col_total = "All intrinsic (z=0) [Gpc^-3 yr^-1]"

col_without_ce_1 = "channel V without CE intrinsic (z=0)"
col_without_ce_2 = "channel II intrinsic (z=0)"

col_with_ce_1 = "channel V with CE intrinsic (z=0)"
col_with_ce_2 = "channel I intrinsic (z=0)"
col_with_ce_3 = "channel III intrinsic (z=0)"
col_with_ce_4 = "channel IV intrinsic (z=0)"


# --------------------------------------------------
# Helper to extract variation ID from filename
# --------------------------------------------------
def extract_variation_id(filepath: str) -> str:
    """
    Example:
    Formation_Channels_Local_Rates_BHBH_xyz_111.csv -> 111
    """
    base = os.path.basename(filepath)
    stem = os.path.splitext(base)[0]
    return stem.split("_")[-1]


# --------------------------------------------------
# Read and process one file
# --------------------------------------------------
def process_one_file(filepath: str, dco_label: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)

    required_cols = [
        col_model,
        col_total,
        col_without_ce_1,
        col_without_ce_2,
        col_with_ce_1,
        col_with_ce_2,
        col_with_ce_3,
        col_with_ce_4,
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {filepath}: {missing}")

    variation_id = extract_variation_id(filepath)

    out = df.copy()

    # Rename model column for clarity
    out = out.rename(columns={col_model: "model_name"})

    # Absolute total rate
    out["total_rate"] = pd.to_numeric(out[col_total], errors="coerce")

    # These channel sums are already fractions in these files
    out["without_CE_fraction"] = (
        pd.to_numeric(out[col_without_ce_1], errors="coerce").fillna(0.0)
        + pd.to_numeric(out[col_without_ce_2], errors="coerce").fillna(0.0)
    )

    out["with_CE_fraction"] = (
        pd.to_numeric(out[col_with_ce_1], errors="coerce").fillna(0.0)
        + pd.to_numeric(out[col_with_ce_2], errors="coerce").fillna(0.0)
        + pd.to_numeric(out[col_with_ce_3], errors="coerce").fillna(0.0)
        + pd.to_numeric(out[col_with_ce_4], errors="coerce").fillna(0.0)
    )

    # Metadata
    out["DCOlabel"] = dco_label
    out["SFRD_variation"] = variation_id
    out["source_file"] = os.path.basename(filepath)

    # Consistency checks
    out["sum_CE_fractions"] = out["with_CE_fraction"] + out["without_CE_fraction"]
    out["sum_minus_one"] = out["sum_CE_fractions"] - 1.0
    out["fractions_sum_to_one"] = np.isclose(
        out["sum_CE_fractions"],
        1.0,
        rtol=1e-6,
        atol=1e-8
    )

    # Keep only the useful columns
    out = out[
        [
            "DCOlabel",
            "SFRD_variation",
            "source_file",
            "model_name",
            "total_rate",
            "without_CE_fraction",
            "with_CE_fraction",
            "sum_CE_fractions",
            "sum_minus_one",
            "fractions_sum_to_one",
        ]
    ]

    return out


# --------------------------------------------------
# Process one DCO label
# --------------------------------------------------
def build_combined_sfrd_file(
    dco_label: str,
    input_dir: str,
    output_dir: str,
) -> pd.DataFrame:
    """
    Build one combined CSV for one DCO label, e.g. BHBH, BHNS, NSNS.
    """
    file_pattern = os.path.join(
        input_dir,
        f"Formation_Channels_Local_Rates_{dco_label}_xyz_*.csv"
    )

    output_csv = os.path.join(
        output_dir,
        f"Broekgaarden_{dco_label}_all_SFRD_variations_combined.csv"
    )

    all_files = sorted(glob.glob(file_pattern))

    if len(all_files) == 0:
        raise FileNotFoundError(f"No files found matching: {file_pattern}")

    print(f"\n==============================")
    print(f"Processing {dco_label}")
    print(f"Found {len(all_files)} files")

    all_dfs = []
    for f in all_files:
        print(f"Processing: {os.path.basename(f)}")
        all_dfs.append(process_one_file(f, dco_label=dco_label))

    combined = pd.concat(all_dfs, ignore_index=True)

    # Optional: sort nicely
    combined["variation_sort_key"] = combined["SFRD_variation"].astype(str)
    combined = combined.sort_values(["variation_sort_key", "model_name"]).drop(columns="variation_sort_key")

    # Summary checks
    n_bad = (~combined["fractions_sum_to_one"]).sum()
    print(f"\nRows where with_CE_fraction + without_CE_fraction != 1 for {dco_label}: {n_bad}")

    if n_bad > 0:
        print(
            combined.loc[~combined["fractions_sum_to_one"], [
                "SFRD_variation", "model_name",
                "with_CE_fraction", "without_CE_fraction",
                "sum_CE_fractions", "sum_minus_one"
            ]].head(20).to_string(index=False)
        )

    # Save output
    combined.to_csv(output_csv, index=False)
    print(f"\nWrote combined CSV to:\n{output_csv}")

    return combined


# --------------------------------------------------
# Run for one or more DCO labels
# --------------------------------------------------
DCOlabels = ["BHBH", "BHNS", "NSNS"]

all_outputs = {}
for dco_label in DCOlabels:
    try:
        all_outputs[dco_label] = build_cmbined_sfrd_file(
            dco_label=dco_label,
            input_dir=input_dir,
            output_dir=output_dir,
        )
    except FileNotFoundError as e:
        print(f"\nSkipping {dco_label}: {e}")


Processing BHBH
Found 28 files
Processing: Formation_Channels_Local_Rates_BHBH_xyz_000.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_111.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_112.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_113.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_121.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_122.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_123.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_131.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_132.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_133.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_211.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_212.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_213.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_221.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_222.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_223.csv
Processi

In [12]:
# import os
# import glob
# import pandas as pd
# import numpy as np

# # --------------------------------------------------
# # Paths
# # --------------------------------------------------
# input_dir = "/Users/floorbroekgaarden/Projects/GitHub/DCO_FormationChannels/dataFiles/data_Fig_1"
# file_pattern = os.path.join(input_dir, "Formation_Channels_Local_Rates_BHBH_xyz_*.csv")
# output_dir ="/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/plottingCode/Fig_comparison/Data_formation_channels_intrinsic/extra_fc_rates"

# output_csv = os.path.join(output_dir, "Broekgaarden_BHBH_all_SFRD_variations_combined.csv")
# output_xlsx = os.path.join(output_dir, "Broekgaarden_BHBH_all_SFRD_variations_combined.xlsx")

# # --------------------------------------------------
# # Column names in the input files
# # --------------------------------------------------
# col_model = "formation channel"
# col_total = "All intrinsic (z=0) [Gpc^-3 yr^-1]"

# col_without_ce_1 = "channel V without CE intrinsic (z=0)"
# col_without_ce_2 = "channel II intrinsic (z=0)"

# col_with_ce_1 = "channel V with CE intrinsic (z=0)"
# col_with_ce_2 = "channel I intrinsic (z=0)"
# col_with_ce_3 = "channel III intrinsic (z=0)"
# col_with_ce_4 = "channel IV intrinsic (z=0)"

# # --------------------------------------------------
# # Helper to extract variation ID from filename
# # --------------------------------------------------
# def extract_variation_id(filepath: str) -> str:
#     """
#     Example:
#     Formation_Channels_Local_Rates_BHBH_xyz_111.csv -> 111
#     """
#     base = os.path.basename(filepath)
#     stem = os.path.splitext(base)[0]
#     return stem.split("_")[-1]

# # --------------------------------------------------
# # Read and process one file
# # --------------------------------------------------
# def process_one_file(filepath: str) -> pd.DataFrame:
#     df = pd.read_csv(filepath)

#     required_cols = [
#         col_model,
#         col_total,
#         col_without_ce_1,
#         col_without_ce_2,
#         col_with_ce_1,
#         col_with_ce_2,
#         col_with_ce_3,
#         col_with_ce_4,
#     ]
#     missing = [c for c in required_cols if c not in df.columns]
#     if missing:
#         raise ValueError(f"Missing columns in {filepath}: {missing}")

#     variation_id = extract_variation_id(filepath)

#     out = df.copy()

#     # Rename model column for clarity
#     out = out.rename(columns={col_model: "model_name"})

#     # Compute rates
#     out["total_rate"] = pd.to_numeric(out[col_total], errors="coerce")

#     out["without_CE_fraction"] = (
#         pd.to_numeric(out[col_without_ce_1], errors="coerce").fillna(0.0)
#         + pd.to_numeric(out[col_without_ce_2], errors="coerce").fillna(0.0)
#     )

#     out["with_CE_fraction"] = (
#         pd.to_numeric(out[col_with_ce_1], errors="coerce").fillna(0.0)
#         + pd.to_numeric(out[col_with_ce_2], errors="coerce").fillna(0.0)
#         + pd.to_numeric(out[col_with_ce_3], errors="coerce").fillna(0.0)
#         + pd.to_numeric(out[col_with_ce_4], errors="coerce").fillna(0.0)
#     ) 

#     # Metadata
#     out["SFRD_variation"] = variation_id
#     out["source_file"] = os.path.basename(filepath)

#     # Consistency checks
#     # Sum of fractions (should be ~1)
#     out["sum_CE_fractions"] = out["with_CE_fraction"] + out["without_CE_fraction"]
#     out["sum_minus_one"] = out["sum_CE_fractions"] - 1.0
#     out["fractions_sum_to_one"] = np.isclose(
#         out["sum_CE_fractions"],
#         1.0,
#         rtol=1e-6,
#         atol=1e-8
#     )

#     # Keep only the useful columns
#     out = out[
#         [
#             "SFRD_variation",
#             "source_file",
#             "model_name",
#             "total_rate",
#             "without_CE_fraction",
#             "with_CE_fraction",
#             "sum_CE_fractions",
#             "sum_minus_one",
#             "fractions_sum_to_one",
#         ]
#     ]

#     return out

# # --------------------------------------------------
# # Process all files
# # --------------------------------------------------
# all_files = sorted(glob.glob(file_pattern))

# if len(all_files) == 0:
#     raise FileNotFoundError(f"No files found matching: {file_pattern}")

# print(f"Found {len(all_files)} files")

# all_dfs = []
# for f in all_files:
#     print(f"Processing: {os.path.basename(f)}")
#     all_dfs.append(process_one_file(f))

# combined = pd.concat(all_dfs, ignore_index=True)

# # Optional: sort nicely
# combined["variation_sort_key"] = combined["SFRD_variation"].astype(str)
# combined = combined.sort_values(["variation_sort_key", "model_name"]).drop(columns="variation_sort_key")

# # --------------------------------------------------
# # Summary checks
# # --------------------------------------------------
# n_bad = (~combined["fractions_sum_to_one"]).sum()
# print(f"\nRows where with_CE_fraction + without_CE_fraction != 1: {n_bad}")

# if n_bad > 0:
#     print(
#         combined.loc[~combined["fractions_sum_to_one"], [
#             "SFRD_variation", "model_name",
#             "with_CE_fraction", "without_CE_fraction",
#             "sum_CE_fractions", "sum_minus_one"
#         ]].head(20).to_string(index=False)
#     )
# # --------------------------------------------------
# # Save outputs
# # --------------------------------------------------
# combined.to_csv(output_csv, index=False)
# print(f"\nWrote combined CSV to:\n{output_csv}")

# # with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
# #     combined.to_excel(writer, sheet_name="all_variations", index=False)

# #     # small summary sheet
# #     summary = (
# #         combined.groupby("SFRD_variation")
# #         .agg(
# #             n_models=("model_name", "count"),
# #             mean_total_rate=("total_rate", "mean"),
# #             mean_without_CE_fraction=("without_CE_fraction", "mean"),
# #             mean_with_CE_fraction=("with_CE_fraction", "mean"),
# #             all_rows_match_total=("sum_matches_total", "all"),
# #         )
# #         .reset_index()
# #         .sort_values("SFRD_variation")
# #     )
# #     summary.to_excel(writer, sheet_name="summary", index=False)

# print(f"Wrote Excel file to:\n{output_xlsx}")

Found 28 files
Processing: Formation_Channels_Local_Rates_BHBH_xyz_000.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_111.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_112.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_113.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_121.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_122.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_123.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_131.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_132.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_133.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_211.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_212.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_213.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_221.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_222.csv
Processing: Formation_Channels_Local_Rates_BHBH_xyz_223.csv
Processing: Formation_Cha

/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/plottingCode/Fig_comparison/Data_formation_channels_intrinsic/extra_fc_rates
